In [1]:
%load_ext autoreload
%autoreload 2

from folde.data import get_proteingym_dataset
dms_id = 'FLIP-AAV'

wt_aa_seq, naturalness_df, embedding_df, activity_df, category_df = get_proteingym_dataset(
    dms_id,
    '300m',
    '600m',
)

/home/jacobroberts/foldy-internal/backend/src/folde/data.py:179: DtypeWarning: Columns (12,14,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  incomplete_activity_df = pd.read_csv(FLIP_AAV_DATA_FILE)
/home/jacobroberts/foldy-internal/backend/src/folde/data.py:222: DtypeWarning: Columns (12,14,16,22) have mixed types. Specify dtype option on import or set low_memory=False.
  category_df = pd.read_csv(FLIP_AAV_DATA_FILE)
/home/jacobroberts/foldy-internal/backend/src/folde/data.py:230: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  category_df = category_df.fillna(False)


In [2]:
from folde.few_shot_models import get_few_shot_model

configs = {
    "RandomForestFewShotModel": {
        "n_estimators": 100,
        "criterion": "friedman_mse",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "min_weight_fraction_leaf": 0.0,
        "max_features": 1.0,
        "max_leaf_nodes": None,
        "min_impurity_decrease": 0.0,
        "bootstrap": True,
        "oob_score": False,
        "n_jobs": None,
        "verbose": 0,
        "warm_start": False,
        "ccp_alpha": 0.0,
        "max_samples": None,
        "random_state": 42,
        "wt_aa_seq": wt_aa_seq,
    },
    "TorchMLPFewShotModel": {
        "pretrain": True,
        "pretrain_epochs": 50,
        "ensemble_size": 5,
        "embedding_dim": 960,
        "hidden_dims": [100, 50],
        "dropout": 0.2,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "train_epochs": 200,
        "train_patience": 40,
        "val_frequency": 10,
        "do_validation_with_pair_fraction": 0.2,
        "decision_mode": "constantliar",
        "lie_noise_stddev_multiplier": 2.0,
        "random_state": 42,
        "wt_aa_seq": wt_aa_seq,
    },
}

models = {}
for model_name, model_params in configs.items():
    models[model_name] = get_few_shot_model(model_name, **model_params)

In [3]:
from app.helpers.sequence_util import is_homolog_seq_id, get_loci_set

naturalness_series = naturalness_df.wt_marginal
embedding_series = embedding_df.embedding
activity_series = activity_df.DMS_score

def is_single_mutant_id(seq_id: str) -> bool:
    if seq_id == 'WT' or is_homolog_seq_id(seq_id):
        return False
    return len(get_loci_set(seq_id)) == 1
single_mutant_seq_ids = [seq_id for seq_id in naturalness_series.index if is_single_mutant_id(seq_id)]
pretraining_naturalness_series = naturalness_series.loc[single_mutant_seq_ids]
pretraining_embedding_series = embedding_series.loc[single_mutant_seq_ids]

for model_name, model in models.items():
    model.pretrain(
        pretraining_naturalness_series,
        pretraining_embedding_series,
    )


In [7]:
activity_df.levenshtein_distance

seq_id
M1A               NaN
M1C               NaN
M1D               NaN
M1E               NaN
M1F               NaN
                 ... 
HOM-AAV284004    38.0
HOM-AAV284005    38.0
HOM-AAV284006    39.0
HOM-AAV284007    39.0
HOM-AAV284008    39.0
Name: levenshtein_distance, Length: 297011, dtype: float64

In [8]:
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from folde.util import get_consensus_scores, get_top_percentile_recall_score, top_k_mask
from tqdm.contrib.concurrent import thread_map
from random import sample


def get_single_sim_results(model_name, benchmark=None, max_train_lev_dist=None, test_lev_dist=None):
    single_sim_results = {}

    if max_train_lev_dist and test_lev_dist:
        assert benchmark is None
        benchmark = f'{max_train_lev_dist}-vs-{test_lev_dist}'
        is_train_row = activity_df.levenshtein_distance <= max_train_lev_dist
        is_test_row = activity_df.levenshtein_distance == test_lev_dist
        train_seq_ids = activity_df[is_train_row].index
        test_seq_ids = activity_df[is_test_row].index

    elif benchmark == 'one_vs_two_split':
        is_train_row = category_df['one_vs_many_split']
        is_test_row = category_df['two_vs_many_split'] & (~category_df['one_vs_many_split'])
        train_seq_ids = category_df[is_train_row].index
        test_seq_ids = category_df[is_test_row].index

    elif benchmark == 'one_vs_seven_split':
        is_train_row = category_df['one_vs_many_split']
        is_test_row = category_df['seven_vs_many_split'] & (~category_df['one_vs_many_split'])
        train_seq_ids = category_df[is_train_row].index
        test_seq_ids = category_df[is_test_row].index

    elif benchmark == 'two_vs_seven_split':
        is_train_row = category_df['two_vs_many_split']
        is_test_row = category_df['seven_vs_many_split'] & (~category_df['two_vs_many_split'])
        train_seq_ids = category_df[is_train_row].index
        test_seq_ids = category_df[is_test_row].index

    else:
        is_train_row = category_df[benchmark]
        train_seq_ids = category_df[is_train_row].index
        test_seq_ids = category_df[~is_train_row].index

    train_seq_ids = sample(list(train_seq_ids), min(100, len(train_seq_ids)))

    model.fit(
        naturalness_series.loc[train_seq_ids],
        embedding_series.loc[train_seq_ids],
        activity_series.loc[train_seq_ids],
        None, None, None
    )

    predictions = model.predict(
        naturalness_series.loc[test_seq_ids],
        embedding_series.loc[test_seq_ids],
    )

    mean_prediction = sum(predictions) / len(predictions)

    held_out_activity_series = activity_series.loc[test_seq_ids]
    activity_is_nonnull = held_out_activity_series.notna()
    
    single_sim_results[(model_name, benchmark, 'spearman')] = spearmanr(
        mean_prediction[activity_is_nonnull],
        held_out_activity_series[activity_is_nonnull]
    )
    
    single_sim_results[(model_name, benchmark, 'recall1pct')] = get_top_percentile_recall_score(
        mean_prediction[activity_is_nonnull].to_numpy(),
        held_out_activity_series[activity_is_nonnull].to_numpy(),
        1,
    )
    
    single_sim_results[(model_name, benchmark, 'recall10pct')] = get_top_percentile_recall_score(
        mean_prediction[activity_is_nonnull].to_numpy(),
        held_out_activity_series[activity_is_nonnull].to_numpy(),
        10,
    )
    return single_sim_results


In [10]:
from tqdm.notebook import tqdm
matrix_results = {}
for train_lev_dist in tqdm(range(1, 7), desc='train_lev_dist'):
    for test_lev_dist in tqdm(range(train_lev_dist + 1, 8), desc='test_lev_dist'):
        matrix_results.update(get_single_sim_results(
            model_name='TorchMLPFewShotModel',
            max_train_lev_dist=train_lev_dist,
            test_lev_dist=test_lev_dist,
        ))

matrix_results

train_lev_dist:   0%|          | 0/6 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/6 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/5 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/4 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/3 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/2 [00:00<?, ?it/s]

test_lev_dist:   0%|          | 0/1 [00:00<?, ?it/s]

{('TorchMLPFewShotModel',
  '1-vs-2',
  'spearman'): SignificanceResult(statistic=0.4832120390264099, pvalue=0.0),
 ('TorchMLPFewShotModel', '1-vs-2', 'recall1pct'): 0.05921052631578947,
 ('TorchMLPFewShotModel', '1-vs-2', 'recall10pct'): 0.39585389930898324,
 ('TorchMLPFewShotModel',
  '1-vs-3',
  'spearman'): SignificanceResult(statistic=0.22906526148993686, pvalue=4.789887668258688e-112),
 ('TorchMLPFewShotModel', '1-vs-3', 'recall1pct'): 0.0,
 ('TorchMLPFewShotModel', '1-vs-3', 'recall10pct'): 0.12673056443024494,
 ('TorchMLPFewShotModel',
  '1-vs-4',
  'spearman'): SignificanceResult(statistic=0.20603354027251358, pvalue=3.232461803218161e-94),
 ('TorchMLPFewShotModel', '1-vs-4', 'recall1pct'): 0.0,
 ('TorchMLPFewShotModel', '1-vs-4', 'recall10pct'): 0.13190184049079753,
 ('TorchMLPFewShotModel',
  '1-vs-5',
  'spearman'): SignificanceResult(statistic=0.2474423878566957, pvalue=8.790796450723498e-238),
 ('TorchMLPFewShotModel', '1-vs-5', 'recall1pct'): 0.011627906976744186,
 ('Tor

In [ ]:
results = {}
for i, (model_name, model) in enumerate(tqdm(models.items(), desc="Models", position=0)):
    for j, benchmark in enumerate(tqdm(
        ['one_vs_many_split', 'two_vs_many_split', 'seven_vs_many_split', 'low_vs_high_split', 'mut_des_split'],
        desc="Benchmarks", position=1, leave=False
    )):
        single_sim_results = get_single_sim_results(model_name, benchmark)
        results.update(single_sim_results)

results

In [18]:
get_single_sim_results('TorchMLPFewShotModel', 'one_vs_two_split')

{('TorchMLPFewShotModel',
  'one_vs_two_split',
  'spearman'): SignificanceResult(statistic=0.477678141921578, pvalue=0.0),
 ('TorchMLPFewShotModel',
  'one_vs_two_split',
  'recall1pct'): 0.08552631578947369,
 ('TorchMLPFewShotModel',
  'one_vs_two_split',
  'recall10pct'): 0.4021059559065482}

In [19]:
get_single_sim_results('RandomForestFewShotModel', 'one_vs_two_split')

{('RandomForestFewShotModel',
  'one_vs_two_split',
  'spearman'): SignificanceResult(statistic=0.43682340514858153, pvalue=0.0),
 ('RandomForestFewShotModel',
  'one_vs_two_split',
  'recall1pct'): 0.07236842105263158,
 ('RandomForestFewShotModel',
  'one_vs_two_split',
  'recall10pct'): 0.36031589338598224}

In [21]:
get_single_sim_results('TorchMLPFewShotModel', 'one_vs_seven_split')

{('TorchMLPFewShotModel',
  'one_vs_seven_split',
  'spearman'): SignificanceResult(statistic=0.45318091879764844, pvalue=0.0),
 ('TorchMLPFewShotModel',
  'one_vs_seven_split',
  'recall1pct'): 0.013119533527696793,
 ('TorchMLPFewShotModel',
  'one_vs_seven_split',
  'recall10pct'): 0.1586699722910894}

In [22]:
get_single_sim_results('RandomForestFewShotModel', 'one_vs_seven_split')

{('RandomForestFewShotModel',
  'one_vs_seven_split',
  'spearman'): SignificanceResult(statistic=0.4045459478766662, pvalue=0.0),
 ('RandomForestFewShotModel',
  'one_vs_seven_split',
  'recall1pct'): 0.01020408163265306,
 ('RandomForestFewShotModel',
  'one_vs_seven_split',
  'recall10pct'): 0.14510718973311945}

In [24]:
get_single_sim_results('TorchMLPFewShotModel', 'two_vs_seven_split')

{('TorchMLPFewShotModel',
  'two_vs_seven_split',
  'spearman'): SignificanceResult(statistic=0.23946019395042983, pvalue=0.0),
 ('TorchMLPFewShotModel',
  'two_vs_seven_split',
  'recall1pct'): 0.002617801047120419,
 ('TorchMLPFewShotModel',
  'two_vs_seven_split',
  'recall10pct'): 0.10316836868290129}

In [25]:
get_single_sim_results('RandomForestFewShotModel', 'two_vs_seven_split')

{('RandomForestFewShotModel',
  'two_vs_seven_split',
  'spearman'): SignificanceResult(statistic=0.23316613801160102, pvalue=0.0),
 ('RandomForestFewShotModel',
  'two_vs_seven_split',
  'recall1pct'): 0.002617801047120419,
 ('RandomForestFewShotModel',
  'two_vs_seven_split',
  'recall10pct'): 0.09374181722964127}

In [16]:
# Save results to json file.

# Convert SignificanceResult objects to dictionaries
json_results = {}
for k, v in results.items():
    if hasattr(v, "statistic"):
        json_results[str(k)] = float(v.statistic)
    else:
        json_results[str(k)] = float(v)
# Save to json file with timestamp
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'notebooks/jacob/model_evals/flip_benchmark_results_{timestamp}.json'

import json
with open(filename, 'w') as f:
    json.dump(json_results, f, indent=2)

print(f"Results saved to {filename}")


Results saved to notebooks/jacob/model_evals/flip_benchmark_results_20250722_185021.json
